# OCR Detection — Azure OpenAI

## Activate the packages

In [4]:
import base64
import os
from pathlib import Path

import pymupdf
from openai import AzureOpenAI
from dotenv import load_dotenv

## Load the credential from the .env file

Uses Azure OpenAI keys from `.env`:
- `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_DEPLOYMENT_NAME`
- `AZURE_OPENAI_API_VERSION`

In [6]:
# Always load project-root .env (works even if notebook cwd differs)
ROOT = Path.cwd()
load_dotenv(ROOT / ".env")
print("Loaded .env from:", (ROOT / ".env").resolve() if (ROOT / ".env").exists() else "NOT FOUND")

Loaded .env from: /Users/kaishinglee/Documents/AI Solutions/OCR/.env


In [7]:
PDF_PATH = "test_invoice.pdf"          # <-- change this
MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME") or os.getenv("OCR_MODEL", "gpt-4o")
DPI = int(os.getenv("OCR_DPI", "200"))

print(f"PDF_PATH={PDF_PATH}")
print(f"MODEL (deployment)={MODEL}")
print(f"DPI={DPI}")

PDF_PATH=test_invoice.pdf
MODEL (deployment)=gpt-4o
DPI=200


## Azure OpenAI API Connection

In [9]:
api_key = os.getenv("AZURE_OPENAI_API_KEY")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")

if not api_key or not endpoint:
    raise RuntimeError(
        "Missing AZURE_OPENAI_API_KEY or AZURE_OPENAI_ENDPOINT in .env"
    )

# Azure AI Foundry project URLs (.../api/projects/...) break AzureOpenAI chat;
# use the resource root instead (e.g. https://<resource>.services.ai.azure.com)
if "/api/projects/" in endpoint:
    endpoint = endpoint.split("/api/projects/")[0]
    print("Normalized endpoint (stripped /api/projects/...):", endpoint)

client = AzureOpenAI(
    api_key=api_key,
    azure_endpoint=endpoint,
    api_version=api_version,
)

print(f"Endpoint: {endpoint}")
print(f"API version: {api_version}")
print(f"Deployment: {MODEL}")

Endpoint: https://videcoding2-resource.services.ai.azure.com
API version: 2024-08-01-preview
Deployment: gpt-4o


In [10]:
PROMPT = "Extract all text from this image. Output clean Markdown only."

In [11]:
# ---------- 1. PDF → PNG pages ----------
def pdf_to_png_bytes(pdf_path: str, dpi: int = 200) -> list[bytes]:
    zoom = dpi / 72.0
    matrix = pymupdf.Matrix(zoom, zoom)
    pages = []
    with pymupdf.open(pdf_path) as doc:
        for page in doc:
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            pages.append(pix.tobytes("png"))
    return pages

In [12]:
# ---------- 2. One page OCR ----------
def ocr_png_bytes(png_bytes: bytes) -> str:
    """Send one page image to Azure OpenAI vision and return extracted text."""
    b64 = base64.b64encode(png_bytes).decode()
    data_url = f"data:image/png;base64,{b64}"

    resp = client.chat.completions.create(
        model=MODEL,  # Azure deployment name, e.g. gpt-4o
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT},
                {"type": "image_url", "image_url": {"url": data_url}},
            ],
        }],
    )
    return (resp.choices[0].message.content or "").strip()

## OCR processing

In [14]:
pdf_path = Path(PDF_PATH)
pages = pdf_to_png_bytes(PDF_PATH, DPI)
OUTPUT_FILE = "result.md"               # output file name
results = []
print(f"Pages to OCR: {len(pages)}")

Pages to OCR: 1


In [15]:
for i, png in enumerate(pages, 1):
    print(f"OCR page {i}/{len(pages)} ...")
    text = ocr_png_bytes(png)
    results.append(f"## Page {i}\n\n{text}")

# Save to result.md in the same folder as the PDF
output_path = pdf_path.parent / OUTPUT_FILE
output_path.write_text("\n\n".join(results), encoding="utf-8")
print(f"Saved OCR result to: {output_path}")

OCR page 1/1 ...
Saved OCR result to: result.md
